# ⚔️ CRUSADER — F01-A CASTELLAN-AUDIO
## Analyse & Traitement Audio — Pré-transcription Whisper

> *"Know the enemy before you name it."* — Black Templars

---

**Ce notebook tourne en CPU — aucun GPU requis.**

### Rôle :
- Détecte les silences de l'audio source
- Permet de décider (via viewer) de les conserver ou supprimer
- Produit `audio_clean.mp3` + `silence_map.json` pour F01-B

### Étapes :
1. Montage Google Drive
2. Installation Flask (ffmpeg déjà présent sur Colab)
3. Téléchargement des scripts depuis GitHub
4. Configuration des chemins
5. Validation CUSTOS check-in
6. Lancement du viewer interactif

---
## Étape 1 — Montage Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('[OK] Google Drive monté sur /content/drive')

---
## Étape 2 — Vérification FFmpeg + Installation Flask

In [ ]:
import subprocess, sys

# FFmpeg est déjà présent sur Colab — vérification
r = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
if r.returncode == 0:
    version_line = r.stdout.splitlines()[0]
    print(f'[OK] FFmpeg disponible : {version_line}')
else:
    print('[ATTENTION] FFmpeg introuvable — installation...')
    subprocess.run(['apt-get', 'install', '-y', '-q', 'ffmpeg'], check=True)
    print('[OK] FFmpeg installé')

# Flask
print('Installation Flask...')
r2 = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'flask>=2.0', '-q'],
    capture_output=True, text=True
)
if r2.returncode == 0:
    import flask
    print(f'[OK] Flask {flask.__version__} prêt')
else:
    print('[ERREUR] Flask installation échouée')
    print(r2.stderr)

---
## Étape 3 — Téléchargement des scripts depuis GitHub

In [ ]:
import urllib.request, os

BASE_URL   = 'https://raw.githubusercontent.com/kioka8877-ux/CRUSADER/main'
SCRIPT_URL = f'{BASE_URL}/F01_GRIMALDUS/F01A_CASTELLAN_AUDIO/CODEBASE/crs_f01a.py'
VIEWER_URL = f'{BASE_URL}/F01_GRIMALDUS/F01A_CASTELLAN_AUDIO/CODEBASE/crs_f01a_viewer.html'
CUSTOS_URL = f'{BASE_URL}/CRS_CUSTOS.py'

os.makedirs('/content/crusader', exist_ok=True)

for name, url in [('crs_f01a.py', SCRIPT_URL), ('crs_f01a_viewer.html', VIEWER_URL), ('CRS_CUSTOS.py', CUSTOS_URL)]:
    dest = f'/content/crusader/{name}'
    urllib.request.urlretrieve(url, dest)
    size = os.path.getsize(dest)
    print(f'[OK] {name} ({size} bytes) → {dest}')

print()
print('[OK] Téléchargement terminé')

---
## Étape 4 — Configuration des chemins

**Seule cellule à modifier si besoin.**

In [ ]:
import os

# ════════════════════════════════════════
#  CONFIGURATION — À ADAPTER SI BESOIN
# ════════════════════════════════════════
DRIVE_BASE = '/content/drive/MyDrive/DRIVE_CRUSADER'
PORT       = 5001
# ════════════════════════════════════════

IN_DIR     = os.path.join(DRIVE_BASE, 'F01_GRIMALDUS', 'F01A_CASTELLAN_AUDIO', 'IN')
OUT_DIR    = os.path.join(DRIVE_BASE, 'F01_GRIMALDUS', 'F01A_CASTELLAN_AUDIO', 'OUT')
SCRIPT     = '/content/crusader/crs_f01a.py'
VIEWER     = '/content/crusader/crs_f01a_viewer.html'

os.makedirs(IN_DIR,  exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

audio_in = os.path.join(IN_DIR, 'audio_raw.mp3')

print('Configuration F01-A CASTELLAN-AUDIO')
print(f'  IN          : {IN_DIR}')
print(f'  OUT         : {OUT_DIR}')
print(f'  audio_raw   : {audio_in}')
print(f'  Présent     : {os.path.isfile(audio_in)}')
print(f'  Port viewer : {PORT}')

---
## Étape 5 — Validation CUSTOS check-in

In [ ]:
import subprocess

result = subprocess.run(
    ['python', '/content/crusader/CRS_CUSTOS.py',
     '--frigate', 'F01A',
     '--mode', 'check-in',
     '--drive-base', DRIVE_BASE],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('[ATTENTION] CUSTOS check-in échoué — vérifiez que audio_raw.mp3 est dans IN/')
    print(result.stderr)

---
## Étape 6 — Lancement du viewer interactif

Cliquez sur le lien ngrok généré pour ouvrir le viewer dans votre navigateur.

In [ ]:
import subprocess, threading, time

# Lancement Flask en arrière-plan
cmd = [
    'python', SCRIPT,
    '--input',  IN_DIR,
    '--output', OUT_DIR,
    '--viewer', VIEWER,
    '--port',   str(PORT),
]

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# Attendre que Flask soit prêt
time.sleep(2)

# Tunnel ngrok (déjà disponible sur Colab via colab_utils ou pyngrok)
try:
    from pyngrok import ngrok
    public_url = ngrok.connect(PORT)
    print(f'[OK] Viewer disponible sur : {public_url}')
except ImportError:
    # Fallback : accès local uniquement (Colab port forwarding)
    try:
        from google.colab.output import eval_js
        proxy_url = eval_js(f'google.colab.kernel.proxyPort({PORT})')
        print(f'[OK] Viewer disponible sur : {proxy_url}')
    except Exception:
        print(f'[INFO] Viewer Flask sur http://localhost:{PORT}/')
        print('[INFO] Installez pyngrok pour un accès public : pip install pyngrok')

print()
print('Le serveur tourne. Exécutez la cellule suivante pour arrêter.')

---
## Arrêt du serveur (optionnel)

In [ ]:
try:
    proc.terminate()
    print('[OK] Serveur F01-A arrêté')
except Exception as e:
    print(f'[INFO] Arrêt : {e}')

---
## Étape 7 — Validation CUSTOS check-out

Après avoir validé dans le viewer et produit `audio_clean.mp3`.

In [ ]:
import subprocess

result = subprocess.run(
    ['python', '/content/crusader/CRS_CUSTOS.py',
     '--frigate', 'F01A',
     '--mode', 'check-out',
     '--drive-base', DRIVE_BASE],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('[ATTENTION] CUSTOS check-out échoué — audio_clean.mp3 présent dans OUT/ ?')
    print(result.stderr)